**STEP 1 – Data Preparation**
- โหลด, แปลง, เตรียม Feature

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# 1) โหลด
df = pd.read_csv("dataset/mxmh_survey_results.csv")

# 2) เลือกคอลัมน์ความถี่ของแนวเพลง
freq_cols = [c for c in df.columns if c.startswith("Frequency [")]
if not freq_cols:
    raise ValueError("ไม่พบคอลัมน์ Frequency [...] ในไฟล์")

# 3) แผนที่สตริง → ตัวเลข
freq_map = {"Never": 0, "Rarely": 1, "Sometimes": 2, "Very frequently": 3}
X_freq = df[freq_cols].applymap(lambda x: freq_map.get(x, np.nan)).fillna(0).astype(int)

# 4) Label เป้าหมาย
y = df["Fav genre"].astype(str)

# 5) เก็บรายชื่อแนวเพลงจากคอลัมน์ Frequency ให้สวย ๆ
genres = [c.replace("Frequency [", "").replace("]", "") for c in freq_cols]
genre_col_to_name = dict(zip(freq_cols, genres))
name_to_index = {g: i for i, g in enumerate(genres)}

print(f"รูปข้อมูล: X = {X_freq.shape}, y = {y.shape}")
print("ตัวอย่างแนวเพลง:", genres[:8])
X_freq

**STEP 2 – Explore Bias & Missing**
- ตรวจความสมบูรณ์และสมดุลของข้อมูล

In [ ]:
# Missing
print("Missing rate โดยรวมของ Frequency:", X_freq.isna().mean().mean())
print("Missing rate Fav genre:", y.isna().mean())

# Distribution ของ Fav genre
genre_dist = y.value_counts(normalize=True).sort_values(ascending=False) * 100
print("สัดส่วนแนวเพลง (Fav genre) [%]:")
display(genre_dist.round(2))

# (Opt) กราฟ distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
genre_dist.plot(kind="bar")
plt.title("Fav genre distribution (%)")
plt.ylabel("%")
plt.show()

**STEP 3 – Split Train/Test**

- แบ่งข้อมูลออกเป็น 80/20

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)  # แปลง label เป็นตัวเลข

X_train, X_test, y_train, y_test = train_test_split(
    X_freq.values, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
label_names = le.classes_
print("Train/Test:", X_train.shape, X_test.shape)

**STEP 4 – Define Evaluation Metrics**

- นิยามตัวชี้วัด 3 ตัว (วัด “ความแม่นยำในการแนะนำ”)

In [ ]:
import numpy as np


def metrics_at_k(pred_topk, y_true, k=3):
    """
    pred_topk: list/array ของ index แนวเพลงท็อป K (เช่น [idx0, idx1, idx2])
    y_true   : array ของ label จริง (index)
    """
    # Hit@K
    hits = [int(y_true[i] in pred_topk[i][:k]) for i in range(len(y_true))]
    recall_at_k = float(np.mean(hits))

    # MAP@K (แบบง่าย: ถ้าติดลำดับ r -> คะแนน 1/r)
    ap = []
    for i in range(len(y_true)):
        try:
            rank = list(pred_topk[i][:k]).index(y_true[i]) + 1
            ap.append(1.0 / rank)
        except ValueError:
            ap.append(0.0)
    map_k = float(np.mean(ap))

    # Acc@1
    acc1 = float(
        np.mean([int(y_true[i] == pred_topk[i][0]) for i in range(len(y_true))])
    )
    return acc1, recall_at_k, map_k

**STEP 5 – Baseline (Self-Frequency Ranking)**
- ระบบแนะนำแบบง่ายที่สุด

In [ ]:
def self_top3_indices(x_row):
    # คืน index ของแนวเพลงเรียงจากมาก→น้อย
    return np.argsort(-x_row)[:3]


pred_self_top3 = [self_top3_indices(x) for x in X_test]
acc1_b, r3_b, map3_b = metrics_at_k(pred_self_top3, y_test, k=3)

print(
    f"[Baseline by self-frequency] Acc@1={acc1_b:.3f}  Recall@3={r3_b:.3f}  MAP@3={map3_b:.3f}"
)

pred_self_top3[:5]  # ตัวอย่างผลลัพธ์

**STEP 6 – Logistic Regression (Balanced)**
- โมเดล Machine Learning ที่ลด bias ระหว่างเทรน

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_clf = Pipeline(
    [
        # ข้อมูลเป็นจำนวนเต็มเล็ก ๆ (0-3) ไม่จำเป็นต้อง scale มาก แต่ใส่ไว้ช่วยความเสถียร
        ("scaler", StandardScaler(with_mean=False)),
        (
            "clf",
            LogisticRegression(
                max_iter=500, class_weight="balanced", multi_class="auto"
            ),
        ),
    ]
)

logreg_clf.fit(X_train, y_train)

# Predict proba → Top-3
proba = logreg_clf.predict_proba(X_test)
pred_lr_top3 = [np.argsort(-proba[i])[:3] for i in range(len(y_test))]

acc1_lr, r3_lr, map3_lr = metrics_at_k(pred_lr_top3, y_test, k=3)
print(
    f"[LogReg balanced]       Acc@1={acc1_lr:.3f}  Recall@3={r3_lr:.3f}  MAP@3={map3_lr:.3f}"
)

# logreg_clf.predict_proba(X_test)[:2]  # ตัวอย่างผลลัพธ์

**STEP 7 – k-NN Collaborative Filtering (Debiased)**
- เรียนรู้จาก “เพื่อนบ้านที่คล้ายกัน” + ลดความนิยมแนวฮิต

In [ ]:
# ...existing code...

from sklearn.metrics.pairwise import cosine_similarity

def knn_recommend_topk(x_query, X_train, y_train, k=25, alpha=0.5, beta=0.8, topk=None, use_distance_weight=True):
    """
    x_query: เวกเตอร์ความถี่ของผู้ใช้ (shape: n_genres,)
    topk: ถ้าระบุ ให้คืน top-k indices (ตามลำดับ) มิฉะนั้นคืนทั้งอันดับและ scores
    """
    sims = cosine_similarity(x_query.reshape(1, -1), X_train)[0]
    neigh_idx = np.argsort(-sims)[:k]
    scores = np.zeros(X_train.shape[1], dtype=float)
    for j in neigh_idx:
        w = sims[j] if use_distance_weight else 1.0
        scores += w * X_train[j]
        fav_name = le.inverse_transform([y_train[j]])[0]
        fav_idx = name_to_index.get(fav_name, None)
        if fav_idx is not None:
            scores[fav_idx] += w * beta * 3.0

    # ป้องกัน KeyError ด้วย .get และจัดการกรณีไม่มีค่า
    denom = np.array([(pop_counts.get(g, 0) if pop_counts.get(g, 0) > 0 else 1)**alpha for g in genres])
    scores = scores / denom

    if topk is None:
        return np.argsort(-scores), scores
    return np.argsort(-scores)[:topk], scores

# === เพิ่มบรรทัดต่อไปนี้หลังการประกาศฟังก์ชัน knn_recommend_topk ===

# นับความนิยมแนวเพลงจาก y_train เพื่อใช้ลด popularity bias
pop_counts = Counter(le.inverse_transform(y_train))  # <-- แก้ปัญหา NameError ที่เกิดขึ้น

# สร้าง predictions ของ kNN (top-3) บนชุดทดสอบ
pred_knn_top3 = [
    knn_recommend_topk(X_test[i], X_train, y_train, k=25, alpha=0.5, beta=0.8, topk=3)[0]
    for i in range(len(X_test))
]

# คำนวณ metrics (ใช้ metrics_at_k ที่มีอยู่แล้ว)
acc1_knn, r3_knn, map3_knn = metrics_at_k(pred_knn_top3, y_test, k=3)

# แสดงผล
print(f"[kNN CF debiased]       Acc@1={acc1_knn:.3f}  Recall@3={r3_knn:.3f}  MAP@3={map3_knn:.3f}")

**STEP 8 – Compare & Analyze**
- สรุปตารางเปรียบเทียบและเลือกแนวทางใช้งานจริง

In [ ]:
summary = pd.DataFrame(
    [
        {
            "Model": "Baseline (self freq)",
            "Acc@1": acc1_b,
            "Recall@3": r3_b,
            "MAP@3": map3_b,
        },
        {
            "Model": "LogReg (balanced)",
            "Acc@1": acc1_lr,
            "Recall@3": r3_lr,
            "MAP@3": map3_lr,
        },
        {
            "Model": "kNN CF (debiased)",
            "Acc@1": acc1_knn,
            "Recall@3": r3_knn,
            "MAP@3": map3_knn,
        },
    ]
).round(3)
summary

In [ ]:
# แทน try/except แบบเดิม
if 'normalize_user_freq_dict' not in globals():
    import numpy as np
    def normalize_user_freq_dict(user_freq_dict):
        v = np.zeros(len(genres), dtype=float)
        for g, x in user_freq_dict.items():
            idx = name_to_index.get(g)
            if idx is None:
                continue
            if isinstance(x, str):
                v[idx] = float(freq_map.get(x, 0))
            else:
                v[idx] = float(x)
        return v


**STEP 9**

In [ ]:
import numpy as np


def _label_to_name(c):
    # ถ้ามีตัวแปร label_names (จาก LabelEncoder) จะ map index -> ชื่อแนว
    try:
        return label_names[int(c)]
    except Exception:
        return str(c)


def _normalize_scores(scores, kind="auto"):
    s = np.asarray(scores, dtype=float)
    if kind == "prob":  # LogReg: probability อยู่แล้ว
        return np.clip(s, 0.0, 1.0)
    # baseline/knn หรืออื่น ๆ -> min-max
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-8)


def recommend_with_strategy(
    user_freq_dict, strategy="auto", topk=3, cf_alpha=0.5, names_only=False
):
    topk = max(1, min(topk, len(genres)))
    x = normalize_user_freq_dict(user_freq_dict)
    nonzero = int((x > 0).sum())

    if strategy == "auto":
        strategy = "logreg" if nonzero >= max(3, len(genres) // 4) else "knn"

    if strategy == "self":
        full_scores = x.astype(float)  # เวกเตอร์เต็ม 16 แนว
        kind = "auto"

    elif strategy == "logreg":
        if nonzero == 0:
            full_scores = x.astype(float)  # fallback: ใช้ self
            kind = "auto"
        else:
            proba = logreg_clf.predict_proba([x])[0]  # ความยาว = n_classes
            # map index -> ชื่อแนว
            names = [_label_to_name(c) for c in logreg_clf.classes_]
            # สร้างเวกเตอร์เต็มเรียงตาม 'genres'
            name2pos = {n: i for i, n in enumerate(names)}
            full_scores = np.array(
                [proba[name2pos.get(g, -1)] if g in name2pos else 0.0 for g in genres],
                dtype=float,
            )
            kind = "prob"

    elif strategy == "knn":
        if nonzero == 0:
            x = np.ones_like(x) * 1e-3
        idx, scores = knn_recommend_topk(
            x, X_train, y_train, k=25, alpha=cf_alpha, beta=0.8, topk=len(genres)
        )
        # 'scores' ควรเป็นเวกเตอร์เต็มความยาว = n_genres
        full_scores = np.asarray(scores, dtype=float)
        kind = "auto"

    else:
        raise ValueError("Unknown strategy")

    # normalize ทั้งเวกเตอร์ แล้วค่อยเลือก top-k
    norm_full = _normalize_scores(full_scores, kind=kind)
    order = np.argsort(-norm_full)[:topk]
    out = [(genres[i], float(norm_full[i])) for i in order]
    return [n for n, _ in out] if names_only else out


# ===== ตัวอย่างการใช้งานจริง =====
user_a = {
    "Rock": "Sometimes",
    "Pop": "Sometimes",
    "Metal": "Rarely",
    "Classical": "Very frequently",
    "Hip hop": "Sometimes",
    "Jazz": 1,  # ตัวเลขก็ได้ (1=Rarely)
    "Video game music": 0,  # 0=Never
}
print("AUTO:", recommend_with_strategy(user_a, strategy="auto"))
print("SELF:", recommend_with_strategy(user_a, strategy="self"))
print("LOGREG:", recommend_with_strategy(user_a, strategy="logreg"))
print("KNN:", recommend_with_strategy(user_a, strategy="knn"))